In [21]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request
import codecs

In [22]:
files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\thresholds_test"
fuzz_files = [
    f for f in os.listdir(files_pathname)
    if f.startswith("dump6-fuzz-") and f.endswith(".parquet")
]

fuzz_ids = [f.split('-')[-1].removesuffix('.parquet') for f in fuzz_files]

print(f"Found {len(fuzz_ids)} fuzzing files.")
print("Fuzzing IDs in discovery order:", fuzz_ids)

Found 17 fuzzing files.
Fuzzing IDs in discovery order: ['10', '100', '1000', '1500', '20', '200', '2000', '30', '300', '40', '400', '50', '500', '60', '70', '80', '90']


In [23]:

def hex_to_bytes(h):
    #Convert hex/bytes/string to bytes.
    if h is None or (isinstance(h, float) and np.isnan(h)): 
        return b""
    if isinstance(h, (bytes, bytearray)): 
        return bytes(h)
    s = str(h).strip().replace("0x","").replace(" ","")
    if s == "": 
        return b""
    if len(s) % 2 == 1: 
        s = "0"+s
    try: 
        return bytes.fromhex(s)
    except Exception: 
        print(f"Warning: invalid hex input: {h}")
        return str(h).encode("utf-8", errors="ignore")


def hamming_distance(a: bytes, b: bytes) -> int:
    
    
    len_mismatch = (len(a) != len(b))

    # Pad to same length
    max_len = max(len(a), len(b))
    a_padded = a + b'\x00' * (max_len - len(a))
    b_padded = b + b'\x00' * (max_len - len(b))
    
    # Count differences
    #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
    
    distance = 0
    for byte_a, byte_b in zip(a_padded, b_padded):
        distance += bin(byte_a ^ byte_b).count('1')
    return (distance)
    
    #return (dist, len_mismatch)

In [24]:

def compute_hamming_distances_training(dumps, out_csv, process_per_dump=True):
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    
    records = []
    prev_payload = {}  # {arbitration_id: previous_bytes}
    
    for dump_name, df in dumps:
        d = df.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        
        d = d.sort_values("timestamp", kind="mergesort")
        has_label = "label" in d.columns
        
        for _, row in d.iterrows():
            aid = row["arbitration_id"]
            ts = row["timestamp"]
            lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
            curr_bytes = hex_to_bytes(row["data"])
            
            # Get previous payload for this ID
            prev = prev_payload.get(aid)
            
            if prev is not None:
                # Compute Hamming distance
                dist = hamming_distance(curr_bytes, prev)
                #max_len = max(len(curr_bytes), len(prev))
                #norm_dist = dist / (max_len * 8) if max_len > 0 else 0.0
                
                records.append({
                    "dump": dump_name,
                    "timestamp": ts,
                    "arbitration_id": aid,
                    "payload_len": len(curr_bytes),
                    "ham_dist": dist,
                    #"ham_norm": norm_dist,
                    #"len_mismatch": len_mismatch,  
                    "label": lab
                })
            
            

            # Update previous payload
            prev_payload[aid] = curr_bytes
        #False for the bening so we can compute the distances using all 7 dumps
        if process_per_dump:
            prev_payload.clear()
    
    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(out_csv, index=False)
    print(f" Saved: {out_csv} (rows={len(out_df):,})")
    
    return out_df

In [25]:
def compute_hamming_distances_testing(row, prev_payload):
    
    aid = row.arbitration_id
    lab = int(row.label) if hasattr(row, 'label') and pd.notna(row.label) else 0
    data = row.data
    curr_bytes = hex_to_bytes(data)
    
    # Get previous payload for this ID
    prev = prev_payload.get(aid)
    
    if prev is None:
        prev_payload[aid] = curr_bytes
        return None, lab, prev_payload
    
    
    # Compute Hamming distance
    dist = hamming_distance(curr_bytes, prev)
    #should_update = True
    #min_val = None  #INITIALIZE
    #max_val = None  
    
    
    """if ranges_indexed is not None and aid in ranges_indexed.index:
        min_val = ranges_indexed.at[aid, 'min_hamming']
        max_val = ranges_indexed.at[aid, 'max_hamming']
    
    if (min_val is not None) and (max_val is not None):
        if min_val <= dist <= max_val:
            should_update = True  #inside range
        else:
            should_update = False
            
    if should_update:"""
    prev_payload[aid] = curr_bytes
    #ASK PROFESSOR - what to do if the previous payload is malicious because you compare a valid one with a malicious that you saved in the payload_prev and you get a FP????
    return dist, lab, prev_payload

In [26]:
def compute_hamming_distances_with_attack_attribution(row, prev_payload, ranges_indexed, 
                                                       recently_attacked_ids, 
                                                       attribution_window=3):
    """
    When an attack occurs, attribute following high-distance packets to that attack.
    This prevents false positives from state corruption.
    """
    current_aid = row.arbitration_id
    curr_bytes = hex_to_bytes(row.data)
    lab = int(row.label) if hasattr(row, 'label') and pd.notna(row.label) else 0
    
    prev = prev_payload.get(current_aid)
    
    if prev is None:
        prev_payload[current_aid] = curr_bytes
        return None, lab, False, prev_payload, recently_attacked_ids
    
    # Calculate distance
    dist = hamming_distance(curr_bytes, prev)
    attributed_to_attack = False
    
    # Check if this ID was recently attacked
    if current_aid in recently_attacked_ids:
        packets_since = recently_attacked_ids[current_aid]
        
        if packets_since < attribution_window:
            # Within attribution window
            if lab == 0:
                # This is a NORMAL packet, but we'll attribute it to the attack
                attributed_to_attack = True
                # DON'T change dist! Keep the high distance so it gets detected
                
            recently_attacked_ids[current_aid] += 1
            
            if packets_since >= attribution_window:
                del recently_attacked_ids[current_aid]
        else:
            del recently_attacked_ids[current_aid]
    
    # If current packet is an attack, mark this ID
    if lab == 1:
        recently_attacked_ids[current_aid] = 0
    
    # Update for next comparison
    prev_payload[current_aid] = curr_bytes
    
    return dist, lab, attributed_to_attack, prev_payload, recently_attacked_ids

In [27]:

def build_hamming_range_model(hamming_csv, output_csv="artifacts/hamming_ranges_fuzz.csv"):
    
    
    df = pd.read_csv(hamming_csv)
    
    # Group by ID and compute min/max
    ranges = (df.groupby('arbitration_id')['ham_dist']
            .agg(['min', 'max', 'count'])
            .reset_index())
    
    ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
    
    # Compute range size
    ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
    
    # Normalized versions (for 8-byte payloads, max=8)
    ranges['min_norm'] = ranges['min_hamming'] / 64.0
    ranges['max_norm'] = ranges['max_hamming'] / 64.0
    ranges['range_norm'] = ranges['range_size'] / 64.0
    
    # Classify IDs (paper uses sigma=6 for byte-level Hamming)
    sigma = 6 * 4
    """def classify(r):
        if r == 0:
            return 'NoRange'
        elif r <= sigma:
            return 'SmallRange'
        else:
            return 'MidRange'
    
    ranges['class'] = ranges['range_size'].apply(classify)
    
    # Expected detection rates 
    def expected_tpr(cls):
        if cls == 'NoRange':
            return 0.98  
        elif cls == 'SmallRange':
            return 0.97  
        else:
            return 0.25  
    
    ranges['expected_tpr'] = ranges['class'].apply(expected_tpr)
    
    # Sort by range size
    ranges = ranges.sort_values('range_size')"""
    
    # Save
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    ranges.to_csv(output_csv, index=False)
    
    # Print summary
    print(f"\nLearned ranges for {len(ranges)} unique IDs")
    print(f"   Saved to: {output_csv}")

    
    print("="*70)
    return ranges


In [28]:

def detect_simple(dist, arb_id, label, ranges_lookup):
    """
    Simple detection: Flag if distance is OUTSIDE [min, max] range.
    No windowing - direct per-message detection.
    
    Args:
        test_csv: Path to test data with Hamming distances
        ranges_csv: Path to learned ranges
        output_csv: Where to save results
        
    Returns:
        DataFrame with detection results and performance metrics
    """
    #check_anomaly_and_labels = []
    # Load data
    if arb_id not in ranges_lookup:
        return {
            'arbitration_id': arb_id,
            'hamming_distance': dist,
            'detected': True,  # Unknown ID = anomaly
            'label': label,
            'reason': 'unknown_id'
        }
    
    min_ham, max_ham = ranges_lookup[arb_id]
    detected = (dist < min_ham) or (dist > max_ham)
    
    return {
        'arbitration_id': arb_id,
        'hamming_distance': dist,
        'detected': detected,
        'label': label,
        'min_range': min_ham,
        'max_range': max_ham
    }

    
    # Core detection logic from paper:
    # Anomaly if distance is OUTSIDE [min, max]
    detected = (dist < min_ham) or (dist > max_ham)
    
    
    return {
        'arbitration_id': arb_id,
        'hamming_distance': dist,
        'detected': detected,
        'label': label,
        'min_range': min_ham,
        'max_range': max_ham
    }
    """#-----------------
    #-----------------
    #-----------------
    # Check where your FPs are coming from:
    fps = test_with_ranges[
        (test_with_ranges['label'] == 0) & 
        (test_with_ranges['is_anomaly'] == True)
    ]

    print(f"False Positives: {len(fps)}")
    print(f"\nFP Breakdown:")
    print(f"  Length mismatches: {fps['len_mismatch'].sum()}")
    print(f"  Hamming too low: {(fps['ham_dist'] < fps['min_hamming']).sum()}")
    print(f"  Hamming too high: {(fps['ham_dist'] > fps['max_hamming']).sum()}")

    # Check which IDs produce FPs:
    fp_ids = fps.groupby('arbitration_id').size().sort_values(ascending=False)
    print(f"\nTop FP-producing IDs:")
    print(fp_ids.head(10))"""
    
    #-----------------
    #-----------------
    #-----------------
    """
    # Count anomalies
    n_anomalies = test_with_ranges['is_anomaly'].sum()
    n_total = len(test_with_ranges)
    
    print(f"\n Detection Summary:")
    print(f"   Total messages:    {n_total:,}")
    print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in test_with_ranges.columns:
        y_true = test_with_ranges['label'].values
        y_pred = test_with_ranges['is_anomaly'].values
        
        TP = ((y_true == 1) & (y_pred == True)).sum()
        FP = ((y_true == 0) & (y_pred == True)).sum()
        TN = ((y_true == 0) & (y_pred == False)).sum()
        FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\n Detection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
    
    # Save results
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    test_with_ranges.to_csv(output_csv, index=False)
    
    return test_with_ranges, metrics"""


In [29]:
def compute_metrics_from_detections(detections_list):
    """
    Compute metrics from list of detection results.
    Now handles attack attribution for FP reduction.
    """
    if not detections_list:
        return pd.DataFrame(), {}
    
    detections_df = pd.DataFrame(detections_list)
    
    # Count anomalies
    n_anomalies = detections_df['detected'].sum()
    n_total = len(detections_df)
    
    print(f"\nDetection Summary:")
    print(f"   Total messages:    {n_total:,}")
    print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
    
    # Track attribution if available
    if 'attributed_to_attack' in detections_df.columns:
        total_attributed = detections_df['attributed_to_attack'].sum()
        attributed_detected = ((detections_df['attributed_to_attack'] == True) & 
                               (detections_df['detected'] == True)).sum()
        print(f"   Attributed to attack: {total_attributed:,} (detected: {attributed_detected:,})")
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in detections_df.columns and detections_df['label'].notna().any():
        y_true = detections_df['label'].values
        y_pred = detections_df['detected'].values
        
        # Check if attribution is being used
        if 'attributed_to_attack' in detections_df.columns:
            attributed = detections_df['attributed_to_attack'].values
            
            # TP: Real attacks detected OR attributed packets detected
            TP = (((y_true == 1) | (attributed == True)) & (y_pred == True)).sum()
            
            # FP: Normal packets detected that are NOT attributed
            FP = ((y_true == 0) & (attributed == False) & (y_pred == True)).sum()
            
            # TN: Normal packets not detected (excluding attributed)
            TN = ((y_true == 0) & (attributed == False) & (y_pred == False)).sum()
            
            # FN: Real attacks not detected
            FN = ((y_true == 1) & (y_pred == False)).sum()
        else:
            # Original logic without attribution
            TP = ((y_true == 1) & (y_pred == True)).sum()
            FP = ((y_true == 0) & (y_pred == True)).sum()
            TN = ((y_true == 0) & (y_pred == False)).sum()
            FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\nDetection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
    else:
        print("\nNo labels available - cannot compute metrics")
    
    return detections_df, metrics

In [30]:

if __name__ == "__main__":

    
    # Get all fuzz files
    fuzz_files = [f for f in os.listdir(files_pathname) if f.startswith("dump6-fuzz-") and f.endswith(".parquet")]
    
    
    fuzz_ids = sorted([f.replace("dump6-fuzz-", "").replace(".parquet", "") for f in fuzz_files])
    
    print(f"\n Found {len(fuzz_ids)} fuzzing attack files:")
    print(f"   {fuzz_ids}")
    
    ranges = build_hamming_range_model(
    "artifacts/benign_hamming.csv",
    "artifacts/hamming_ranges_fuzz_copy.csv"
)
    print("ranges created")
    ranges_df = pd.read_csv("artifacts/hamming_ranges_fuzz_copy.csv")
    # ========================================================================
    ranges_indexed = ranges.set_index('arbitration_id')
    results_summary = []
    #i made this dict to speed up the lookup of ranges instead of querying the dataframe each time cause it took 4h per file 
    ranges_lookup = {
    row['arbitration_id']: (row['min_hamming'], row['max_hamming'])
    for _, row in ranges_df.iterrows()
}
    
    for i, fuzz_id in enumerate(fuzz_ids, 1):  
        print(f"\n[{i}/{len(fuzz_ids)}] Testing: dump6-fuzz-{fuzz_id}")  
        #just to know at which row i am rn
        #j = 0
        try:
            # Load attack file
            attack_file = os.path.join(files_pathname, f"dump6-fuzz-{fuzz_id}.parquet") 
            attack_df = pq.read_table(attack_file).to_pandas()
            
            print(f"   Loaded: {len(attack_df):,} messages")
            prev_payload = {}
            attack_dump = attack_df.copy()
            
            #populating the prev_payload with the benign data at the start _ASK PROFESSOR IF WISE TO DO THAT
            """print("   Seeding baseline...")
            dump6_sorted = dump6.sort_values("timestamp")
            for _, row in dump6_sorted.iterrows():
                _, _, prev_payload = compute_hamming_distances_testing(row, prev_payload)
            
            print(f"   Baseline seeded: {len(prev_payload)} IDs")"""
            
            #-----------------------------------
            
            
            if "timestamp" not in attack_dump.columns:
                if attack_dump.index.name == "timestamp":
                   attack_dump = attack_dump.reset_index()
                else:
                    attack_dump = attack_dump.reset_index().rename(columns={"index": "timestamp"})
                    
            attack_dump = attack_dump.sort_values("timestamp", kind="mergesort")
            
            detect_list = []
            #prev_payload= {}
            
            recently_attacked_ids = {}  # Track recently attacked IDs
            # Compute Hamming
            for row in attack_dump.itertuples():
                
                #debugging progress
                """if idx % 10000 == 0:
                    print(f"\r   Processing packets... {idx:,}/{len(attack_df):,}", end="", flush=True)"""
                
                #now here it should return just the distance and the label
                current_aid = row.arbitration_id
                attack_ham_dist, lab, prev_payload = compute_hamming_distances_testing(
                    row, 
                    prev_payload,
                    #ranges_indexed,
                    #recently_attacked_ids,
                    #attribution_window=1
                )
                
                if attack_ham_dist is not None:
                    results = detect_simple(
                        attack_ham_dist,
                        current_aid,
                        lab,  
                        ranges_lookup
                    )
                    #results['is_forgiven'] = is_forgiven  # Add forgiveness flag
                    detect_list.append(results)
                """attack_ham_dist, lab, prev_payload = compute_hamming_distances_testing(
                    row, 
                    prev_payload,
                    ranges_indexed # Pass current state
                )
            
                # Detect
                if attack_ham_dist is not None:
                    results = detect_simple(
                        attack_ham_dist,
                        current_aid,
                        lab,  
                        ranges_lookup
                    )
                    detect_list.append(results)"""
                    
                 # Compute metrics for this attack file
            detections_df, metrics = compute_metrics_from_detections(detect_list)
            # Track forgiveness statistics for this file
            #total_forgiven = detections_df['is_forgiven'].sum()
            #forgiven_normal = ((detections_df['label'] == 0) & (detections_df['is_forgiven'] == True)).sum()

            #print(f"   Forgiven packets: {total_forgiven:,} (normal: {forgiven_normal:,})")
                        
            # Save detections
            output_csv = f"artifacts/detections_fuzz_{fuzz_id}.csv"
            Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
            detections_df.to_csv(output_csv, index=False)
            # Store results
            
            
            if 'overall' in metrics:
                
                # Get attribution stats if available
                if 'attributed_to_attack' in detections_df.columns:
                    total_attributed = detections_df['attributed_to_attack'].sum()
                    attributed_detected = ((detections_df['attributed_to_attack'] == True) & 
                                        (detections_df['detected'] == True)).sum()
                else:
                    total_attributed = 0
                    attributed_detected = 0
                results_summary.append({
                    'fuzz_id': fuzz_id,  
                    'n_messages': len(attack_df),
                    'n_attacks': metrics['overall']['TP'] + metrics['overall']['FN'],
                    'TPR': metrics['overall']['TPR'] * 100,
                    'FPR': metrics['overall']['FPR'] * 100,
                    'Precision': metrics['overall']['Precision'] * 100,
                    'F1': metrics['overall']['F1'] * 100,
                    'TP': metrics['overall']['TP'],
                    'FP': metrics['overall']['FP'],
                    'TN': metrics['overall']['TN'],
                    'FN': metrics['overall']['FN'],
                    'attributed_total': total_attributed,
                    'attributed_detected': attributed_detected
                })
                
                """print(f"   TPR: {metrics['overall']['TPR']*100:.1f}%, "
                    f"FPR: {metrics['overall']['FPR']*100:.2f}%, "
                    f"F1: {metrics['overall']['F1']*100:.1f}%")"""
        
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ========================================================================
    # STEP 4: Summary
    # ========================================================================
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv("artifacts/all_fuzz_summary.csv", index=False)
    
    print("\n" + "="*70)
    print("FUZZING ATTACKS - FINAL SUMMARY")
    print("="*70)
    print(f"\nTested: {len(summary_df)}/{len(fuzz_ids)} attack types")
    
    if len(summary_df) > 0:
        print(f"\nOverall Performance:")
        print(f"   Mean TPR:       {summary_df['TPR'].mean():.2f}%")
        print(f"   Median TPR:     {summary_df['TPR'].median():.2f}%")
        print(f"   Min TPR:        {summary_df['TPR'].min():.2f}%")
        print(f"   Max TPR:        {summary_df['TPR'].max():.2f}%")
        print(f"   Mean FPR:       {summary_df['FPR'].mean():.4f}%")
        print(f"   Mean F1:        {summary_df['F1'].mean():.2f}%")
        
        print("\nResults by Fuzzing Intensity:")
        print(summary_df[['fuzz_id', 'n_attacks', 'TPR', 'FPR', 'F1']].to_string(index=False))
        # Forgiveness impact summary
        


 Found 17 fuzzing attack files:
   ['10', '100', '1000', '1500', '20', '200', '2000', '30', '300', '40', '400', '50', '500', '60', '70', '80', '90']

Learned ranges for 64 unique IDs
   Saved to: artifacts/hamming_ranges_fuzz_copy.csv
ranges created

[1/17] Testing: dump6-fuzz-10
   Loaded: 4,289,508 messages

Detection Summary:
   Total messages:    4,289,444
   Flagged anomalies: 16,742 (0.39%)

Detection Results:
   TPR: 89.04% | FPR: 0.19% | F1: 64.90%
   TP: 8,547 | FP: 8,195 | TN: 4,271,650 | FN: 1,052

[2/17] Testing: dump6-fuzz-100
   Loaded: 4,375,908 messages

Detection Summary:
   Total messages:    4,375,844
   Flagged anomalies: 156,954 (3.59%)

Detection Results:
   TPR: 89.17% | FPR: 1.67% | F1: 67.68%
   TP: 85,602 | FP: 71,352 | TN: 4,208,493 | FN: 10,397

[3/17] Testing: dump6-fuzz-1000
   Loaded: 5,239,908 messages

Detection Summary:
   Total messages:    5,239,844
   Flagged anomalies: 1,290,749 (24.63%)

Detection Results:
   TPR: 89.19% | FPR: 10.15% | F1: 76.08